# Generative Art avec Claude Skills

Ce notebook reprend la structure de `lesson_5.ipynb` et l'adapte au skill **`algorithmic-art`**.
Au lieu de produire un fichier Word ou des questions de cours, Claude va générer une **œuvre d'art génératif interactive** (HTML + p5.js) à partir d'un simple brief.

**Ce que tu vas apprendre :**
1. Uploader un skill local (un dossier) via `files_from_dir`
2. Demander à Claude de l'utiliser dans un conteneur d'exécution de code
3. Récupérer l'artefact produit (un fichier HTML auto-suffisant)
4. Le **prévisualiser directement dans le notebook**

On enchaîne deux cas d'usage de difficulté croissante :

| Partie | Entrée | Sortie |
|---|---|---|
| **1. Brief poétique** | Texte court | Œuvre génératrice interactive |
| **2. Data-driven art** | `retail_sales.csv` | Œuvre dont le rythme reflète la série temporelle |

---
## Setup

In [ ]:
from dotenv import load_dotenv
import anthropic
from anthropic.lib import files_from_dir
from IPython.display import IFrame, HTML, display

_ = load_dotenv()

In [ ]:
client = anthropic.Anthropic()

In [ ]:
# Petit helper pour afficher joliment les blocs de réponse
BOLD, RESET = "\033[1m", "\033[0m"
BLUE, GREEN, YELLOW, CYAN, MAGENTA = (
    "\033[94m", "\033[92m", "\033[93m", "\033[96m", "\033[95m"
)

def pretty_print(response):
    """Affiche chaque bloc de la réponse avec un code couleur lisible."""
    for block in response.content:
        if block.type == "text":
            print(f"{BOLD}{BLUE}[TEXT]{RESET}")
            print(block.text, "\n")
        elif block.type == "server_tool_use":
            print(f"{BOLD}{GREEN}[SERVER_TOOL_USE]{RESET}")
            preview = block.input.copy() if isinstance(block.input, dict) else block.input
            if isinstance(preview, dict) and "file_text" in preview and len(preview["file_text"]) > 120:
                preview["file_text"] = preview["file_text"][:120] + " ..."
            print(preview, "\n")
        elif block.type == "text_editor_code_execution_tool_result":
            print(f"{BOLD}{YELLOW}[EDITOR_RESULT]{RESET}")
            s = str(block.content)
            print((s[:120] + " ...") if len(s) > 120 else s, "\n")
        elif block.type == "bash_code_execution_tool_result":
            print(f"{BOLD}{CYAN}[BASH_RESULT]{RESET}")
            print(block.content, "\n")
        else:
            print(f"{BOLD}{MAGENTA}[{block.type.upper()}]{RESET}", "\n")

---
## Partie 1 : « Liquid Light » — art génératif depuis un brief

Le skill `algorithmic-art` produit *deux* artefacts :
1. Un fichier **`.md`** décrivant la **philosophie algorithmique** (le manifesto de l'œuvre).
2. Un fichier **`.html`** auto-suffisant qui contient p5.js, l'algorithme et un panneau de contrôle.

On commence par déposer le skill sur l'API.

### Étape 1 — Uploader le skill `algorithmic-art`

In [ ]:
skill = client.beta.skills.create(
    display_title="Algorithmic Art",
    files=files_from_dir("./custom_skills/algorithmic-art"),
    betas=["skills-2025-10-02"]
)

print(f"Skill ID: {skill.id}")

### Étape 2 — Vérifier qu'il est bien dispo côté API

In [ ]:
skills = client.beta.skills.list(source="custom")
for sk in skills:
    print(f"ID: {sk.id} | Title: {sk.display_title}")

### Étape 3 — Le brief

Un *brief* court suffit — le skill se charge d'écrire une philosophie complète à partir de ces quelques mots, puis de la traduire en code.

> Astuce : reste **suggestif** plutôt que prescriptif. « lumière liquide au crépuscule » donnera de meilleurs résultats que « dessine 1000 cercles bleus ».

In [ ]:
brief = """
Create an algorithmic art piece inspired by 'liquid light at twilight':
flow fields, bioluminescent particles, the slow exhale of an ocean breathing.

Follow the SKILL.md workflow:
  1. Write the algorithmic philosophy as a .md file.
  2. Build the interactive HTML viewer (using templates/viewer.html as the base).

Copy BOTH final files into $OUTPUT_DIR so they can be downloaded.
"""

### Étape 4 — Envoyer la requête

On attache le skill au conteneur d'exécution et on active le tool `code_execution_20250825` — c'est lui qui permet à Claude d'écrire/déplacer les fichiers vers `$OUTPUT_DIR`.

In [ ]:
response = client.beta.messages.create(
    model="claude-sonnet-4-5-20250929",
    max_tokens=16384,
    betas=["code-execution-2025-08-25", "skills-2025-10-02", "files-api-2025-04-14"],
    container={
        "skills": [
            {"type": "custom", "skill_id": skill.id, "version": "latest"}
        ]
    },
    messages=[{
        "role": "user",
        "content": [{"type": "text", "text": brief}]
    }],
    tools=[{
        "type": "code_execution_20250825",
        "name": "code_execution"
    }]
)

Inventaire rapide des blocs reçus (utile pour comprendre ce que Claude a fait dans son conteneur) :

In [ ]:
for block in response.content:
    print(block.type)

In [ ]:
pretty_print(response)

### Étape 5 — Récupérer les artefacts

Quand Claude copie un fichier vers `$OUTPUT_DIR` via `bash`, l'API retourne un `file_id` dans le bloc `bash_code_execution_tool_result`. On les collecte tous puis on télécharge.

In [ ]:
def collect_file_ids(response):
    """Retourne la liste des file_id émis par les blocs bash dans l'ordre."""
    ids = []
    for block in response.content:
        if block.type == "bash_code_execution_tool_result":
            content = getattr(block, "content", None)
            if hasattr(content, "content"):
                for rb in content.content:
                    fid = getattr(rb, "file_id", None)
                    if fid:
                        ids.append(fid)
    return ids

file_ids = collect_file_ids(response)
print("Files generated:", file_ids)

In [ ]:
# On télécharge tous les fichiers et on déduit leur extension du contenu.
# Heuristique simple : si ça commence par '<!DOCTYPE' ou '<html', c'est l'HTML.
import pathlib

saved = []
for i, fid in enumerate(file_ids):
    fc = client.beta.files.download(file_id=fid, betas=["files-api-2025-04-14"])
    raw = fc.read()
    head = raw[:60].lstrip().lower()
    ext = ".html" if head.startswith((b"<!doctype", b"<html")) else ".md"
    out = pathlib.Path(f"liquid_light_{i}{ext}")
    out.write_bytes(raw)
    saved.append(out)
    print(f"  saved {out}  ({len(raw):,} bytes)")

html_files = [p for p in saved if p.suffix == ".html"]
md_files = [p for p in saved if p.suffix == ".md"]

### Étape 6 — Prévisualiser l'œuvre

On lit d'abord la philosophie (le manifesto), puis on charge le HTML interactif dans un `IFrame`.

In [ ]:
if md_files:
    from IPython.display import Markdown
    display(Markdown(md_files[0].read_text(encoding="utf-8")))
else:
    print("(Pas de fichier markdown trouvé — Claude a peut-être tout intégré dans le HTML)")

In [ ]:
if html_files:
    display(IFrame(src=str(html_files[0]), width="100%", height=720))
else:
    print("(Pas de fichier HTML trouvé — vérifie le contenu de saved)")

### Étape 7 — Nettoyage (optionnel)

Le skill restera accessible côté API jusqu'à suppression. On garde la version uploadée pour la Partie 2.

---
## Partie 2 : Data-driven art — laisser une série temporelle façonner l'algorithme

Maintenant, on **branche des données** sur le skill. L'idée : Claude lit `retail_sales.csv`, en extrait le **rythme** (saisonnalité, tendance, volatilité), puis traduit ce rythme en *paramètres* de l'algorithme génératif. L'œuvre ne **trace pas** les données — elle les **respire**.

C'est exactement le pattern de la Partie 2 de `lesson_5.ipynb`, sauf qu'au lieu de produire un rapport `.docx`, on produit une œuvre.

### Étape 1 — Uploader le fichier d'entrée

In [ ]:
file_object = client.beta.files.upload(
    file=open("./data/retail_sales.csv", "rb"),
)
print("Uploaded file id:", file_object.id)

### Étape 2 — Un brief plus subtil

On précise que la donnée est un **germe conceptuel** (« conceptual seed », terme utilisé dans le SKILL.md), pas un sujet de visualisation.

In [ ]:
query = """
The attached CSV is a monthly retail sales time series.

Use the data's RHYTHM as the conceptual seed for an original algorithmic art piece:
  - its 12-month seasonality should become a pulse, a breath, a recurrence
  - its rising trend should shape growth, accretion, or upward drift
  - its noise should feed micro-perturbations, never raw randomness

Do NOT plot the data literally. The artwork should *feel* like the data
to someone who knows it, without revealing its source to anyone else.

Follow the SKILL.md workflow (philosophy .md + interactive .html based on templates/viewer.html).
Copy BOTH final files into $OUTPUT_DIR.
"""

In [ ]:
response2 = client.beta.messages.create(
    model="claude-sonnet-4-5-20250929",
    max_tokens=16384,
    betas=["code-execution-2025-08-25", "skills-2025-10-02", "files-api-2025-04-14"],
    container={
        "skills": [
            {"type": "custom", "skill_id": skill.id, "version": "latest"}
        ]
    },
    messages=[{
        "role": "user",
        "content": [
            {"type": "text", "text": query},
            {"type": "container_upload", "file_id": file_object.id}
        ]
    }],
    tools=[{
        "type": "code_execution_20250825",
        "name": "code_execution"
    }]
)

In [ ]:
for block in response2.content:
    print(block.type)

In [ ]:
pretty_print(response2)

### Étape 3 — Récupérer & prévisualiser l'œuvre data-driven

In [ ]:
import pathlib

file_ids2 = collect_file_ids(response2)
print("Files generated:", file_ids2)

saved2 = []
for i, fid in enumerate(file_ids2):
    fc = client.beta.files.download(file_id=fid, betas=["files-api-2025-04-14"])
    raw = fc.read()
    head = raw[:60].lstrip().lower()
    ext = ".html" if head.startswith((b"<!doctype", b"<html")) else ".md"
    out = pathlib.Path(f"retail_pulse_{i}{ext}")
    out.write_bytes(raw)
    saved2.append(out)
    print(f"  saved {out}  ({len(raw):,} bytes)")

html2 = [p for p in saved2 if p.suffix == ".html"]
md2 = [p for p in saved2 if p.suffix == ".md"]

In [ ]:
from IPython.display import Markdown
if md2:
    display(Markdown(md2[0].read_text(encoding="utf-8")))

In [ ]:
if html2:
    display(IFrame(src=str(html2[0]), width="100%", height=720))

---
## Nettoyage final

On supprime maintenant le skill côté API (toutes ses versions, puis le skill lui-même), comme dans `lesson_5.ipynb`.

In [ ]:
versions = client.beta.skills.versions.list(
    skill_id=skill.id,
    betas=["skills-2025-10-02"]
)
for v in versions.data:
    client.beta.skills.versions.delete(
        skill_id=skill.id,
        version=v.version,
        betas=["skills-2025-10-02"]
    )
print("Versions deleted.")

In [ ]:
client.beta.skills.delete(
    skill_id=skill.id,
    betas=["skills-2025-10-02"]
)

---
## Pour aller plus loin

- **Change le brief** : essaie « cathedral light through stained glass », « starling murmurations », « ferns unfurling at dawn »… Le skill produit une philosophie différente à chaque fois.
- **Change le seed** dans le HTML généré (panneau latéral) — la même œuvre vit des dizaines de variations.
- **Combine avec d'autres skills** : enchaîne `algorithmic-art` + `docx` pour produire un catalogue PDF d'œuvres avec leur manifesto.
- **Source de données alternative** : remplace `retail_sales.csv` par `city_data.csv` (déjà dans `./data/`) et compare les deux esthétiques.